# Chapter 22 - Zero Shot classifier

The model has to perform a task without seeing examples of that task in the prompt.

In [48]:
from dotenv import load_dotenv
import os 
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("API KEY NOT FOUND")

print("API Key Configured")


API Key Configured


In [49]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],

)

# We will implement what we have learn in the past and will keep applying 

# In this chapter we will apply the concept of try & catch and also will add log files

# Will create multiple functions for it 

In [50]:
# Will create a log file with todays date 

import json
from datetime import datetime

file_name = f"llm_app_{datetime.now().strftime('%Y-%m-%d')}.json"

print(file_name)

llm_app_2026-10-08.json


# SIMPLE ZERO SHOT PROMPT

In [51]:

categories = ["Finance", "Technology", "Sports", "Politics"]

text = "Reliance Industries announced a major investment in renewable energy."

prompt = f"""
Classify the following text into exactly one of these categories:

Categories: {categories}

Text: {text}

Return only the category name.
"""


# SIMPLE ZERO SHOT PROMPT WITH FUNCTIONS

In [52]:
def classify_text(categories, text):

    prompt = f"""
Classify the following text into exactly one of these categories:

Categories: {categories}

Text: {text}

Return only the category name."""

    response = llm.invoke(prompt)

    return response.content
        

In [53]:
categories = [
    "Finance",
    "Technology",
    "Sports",
    "Politics"
]

text = "NVIDIA reported strong growth in demand for its AI chips."

result = classify_text(text, categories)

print("Category:", result)

Category: Technology


# Simple Shot Prompt with better Prompts including Rules

In [54]:
def classify_text(text, categories):

    prompt = f"""
You are a text classification system.

Your task is to classify the text into exactly ONE category.

Allowed categories:
{categories}

Rules:
1. Select only one category.
2. Do not create a new category.
3. Do not explain your answer.
4. Return only the category name.

Text:
{text}
"""

    response = llm.invoke(prompt)

    return response.content.strip()

In [55]:
result = classify_text(
    "Tesla reported higher electric vehicle deliveries.",
    ["Finance", "Technology", "Sports", "Politics"]
)

print(result)

Finance


# Simple Shot Prompt with try and catch 

In [56]:
def classify_text(text, categories):
    prompt = f"""
You are a text classification system.

Your task is to classify the text into exactly ONE category.

Allowed categories:
{categories}

Rules:
1. Select only one category.
2. Do not create a new category.
3. Do not explain your answer.
4. Return only the category name.

Text:
{text}
"""
    try:
        response = llm.invoke(prompt)

        return response.content.strip()
    except Exception as e:
        print("Classification Failed")
        print("Error :", e)
        return None
    

In [57]:
result = classify_text(
    "WHAT ARE YOU DOING MAN ?",
    ["Finance", "Technology", "Sports", "Politics"]
)

print(result)

Sports


In [58]:
# I DON'T KNOW WHAT IS THIS CALLED , MAYBE HALLUCITINATION 

# Simple Shot Prompt with try and catch . Wwill also include logs, will create a different functions for logs and then re use it 

In [59]:
def classify_text(text, categories):

    prompt = f"""
You are a text classification system.

Classify the following text into exactly ONE
of the categories provided below.

Categories:
{categories}

Rules:
1. Select exactly one category.
2. Do not create a new category.
3. Return ONLY the category name.
4. Do not provide an explanation.

Text:
{text}
"""


In [60]:
import time
start_time = time.time()


In [62]:
def classify_text(text, categories):
    start_time = time.time()

    prompt = f"""
You are a text classification system.

Your task is to classify the text into exactly ONE category.

Allowed categories:
{categories}

Rules:
1. Select only one category.
2. Do not create a new category.
3. Do not explain your answer.
4. Return only the category name.

Text:
{text}
"""

    try:
        # ------------------------------------------
        # Call LLM
        # ------------------------------------------
        response = llm.invoke(prompt)
        latency = round(time.time() - start_time, 3)

        # ------------------------------------------
        # Get metadata
        # ------------------------------------------
        metadata = response.response_metadata
        usage = response.usage_metadata

        input_tokens = usage.get("input_tokens")
        output_tokens = usage.get("output_tokens")
        total_tokens = usage.get("total_tokens")

        output_details = usage.get("output_token_details", {})
        reasoning_tokens = output_details.get("reasoning")

        result = response.content.strip()

        # ------------------------------------------
        # Validate classification
        # ------------------------------------------
        if result not in categories:
            raise ValueError(f"Invalid category returned by LLM: {result}")

        log_data = {
            "timestamp": datetime.now().isoformat(),
            "event": "ZERO_SHOT_CLASSIFICATION",
            "request": {
                "text_length": len(text),
                "categories": categories
            },
            "model": {
                "name": metadata.get("model_name"),
                "provider": metadata.get("model_provider")
            },
            "response": {
                "id": metadata.get("id"),
                "finish_reason": metadata.get("finish_reason"),
                "category": result
            },
            "tokens": {
                "input": input_tokens,
                "output": output_tokens,
                "reasoning": reasoning_tokens,
                "total": total_tokens
            },
            "performance": {
                "latency_seconds": latency
            },
            "status": "SUCCESS",
            "error": None
        }

        # ------------------------------------------
        # Save JSON log
        # ------------------------------------------
        with open("zero_shot_log.json", "w") as file:
            json.dump(log_data, file, indent=2)

        return result

    except Exception as e:
        latency = round(time.time() - start_time, 3)

        # ------------------------------------------
        # Create error log
        # ------------------------------------------
        log_data = {
            "timestamp": datetime.now().isoformat(),
            "event": "ZERO_SHOT_CLASSIFICATION",
            "request": {
                "text_length": len(text),
                "categories": categories
            },
            "status": "FAILED",
            "performance": {
                "latency_seconds": latency
            },
            "error": {
                "type": type(e).__name__,
                "message": str(e)
            }
        }

        # ------------------------------------------
        # Save error log
        # ------------------------------------------
        with open("zero_shot_log.json", "w") as file:
            json.dump(log_data, file, indent=2)

        print("\nClassification failed.")
        print("Error:", e)
        return None


categories = [
    "Finance",
    "Technology",
    "Sports",
    "Politics"
]

text = "The Reserve Bank increased interest rates."

result = classify_text(
    text,
    categories
)

print(result)


Finance
